In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import matplotlib.pyplot as plt
import os
import sys
sys.path.append(os.path.abspath('..'))
from src.dataset import RawTokenDataset


In [2]:
class Cfg:
    data_dir = "/root/work/data/raw/train_v1.1"
    window_size = 6
    stride = 1

    # slot separation
    num_slots = 8          # 二分離なら2がまずおすすめ（8は重い＆難しい）
    d_model = 128
    slot_iters = 3

    # decoder
    dec_hidden = 256

    # training
    batch_size = 8
    accum_steps = 64        # 実効64
    num_epochs = 20
    lr = 3e-4
    weight_decay = 1e-4
    grad_clip = 1.0

    # sampling
    pos_sample = 64        # 256からランダムサンプル（まず64）
    neg_samples = 256      # sampled CE の負例数（まず256）
    ce_weight = 0.2        # tokenCEを埋め込み再構成にどれだけ混ぜるか
    emb_loss = "cosine"    # "mse" or "cosine"
    temperature = 0.07     # sampled CE の温度

    detach_slots = True    # ★重要：時間BPTT切る
    device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Cfg()


In [3]:
class SlotAttention(nn.Module):
    def __init__(self, num_slots, dim, iters=3):
        super().__init__()
        self.num_slots = num_slots
        self.iters = iters
        self.scale = dim ** -0.5

        self.slots_mu = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.slots_logsigma = nn.Parameter(torch.zeros(1, 1, dim))

        self.norm_in = nn.LayerNorm(dim)
        self.norm_slots = nn.LayerNorm(dim)
        self.norm_ff = nn.LayerNorm(dim)

        self.to_q = nn.Linear(dim, dim, bias=False)
        self.to_k = nn.Linear(dim, dim, bias=False)
        self.to_v = nn.Linear(dim, dim, bias=False)

        self.gru = nn.GRUCell(dim, dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 2), nn.GELU(), nn.Linear(dim * 2, dim)
        )

    def forward(self, inputs, prev_slots=None):
        # inputs: [B, N, D]
        B, N, D = inputs.shape
        if prev_slots is None:
            mu = self.slots_mu.expand(B, self.num_slots, -1)
            sigma = self.slots_logsigma.expand(B, self.num_slots, -1).exp()
            slots = mu + sigma * torch.randn_like(mu)
        else:
            slots = prev_slots

        x = self.norm_in(inputs)
        k = self.to_k(x)
        v = self.to_v(x)

        for _ in range(self.iters):
            slots_prev = slots
            s = self.norm_slots(slots)
            q = self.to_q(s)

            dots = torch.einsum("bkd,bnd->bkn", q, k) * self.scale
            attn = dots.softmax(dim=1) + 1e-8
            attn = attn / attn.sum(dim=-1, keepdim=True)
            updates = torch.einsum("bnd,bkn->bkd", v, attn)

            slots = self.gru(
                updates.reshape(-1, D),
                slots_prev.reshape(-1, D)
            ).reshape(B, self.num_slots, D)

            slots = slots + self.mlp(self.norm_ff(slots))

        return slots


In [4]:
class SlotEmbeddingDecoder(nn.Module):
    def __init__(self, num_slots, d_model, n_positions, hidden):
        super().__init__()
        self.num_slots = num_slots
        self.n_positions = n_positions

        self.pos = nn.Parameter(torch.randn(1, 1, n_positions, d_model) * 0.02)

        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(),
            nn.Linear(hidden, d_model), nn.GELU(),
        )
        self.mask_head = nn.Linear(d_model, 1)     # -> mask logits
        self.emb_head  = nn.Linear(d_model, d_model)  # -> recon embedding

    def forward(self, slots, pos_idx=None):
        # slots: [B,K,D]
        B, K, D = slots.shape
        if pos_idx is None:
            pos = self.pos                      # [1,1,N,D]
            P = self.n_positions
        else:
            pos = self.pos[:, :, pos_idx, :]    # [1,1,P,D]
            P = pos.shape[2]

        h = slots[:, :, None, :].expand(B, K, P, D) + pos   # [B,K,P,D]
        h = self.mlp(h)
        mask_logits = self.mask_head(h).squeeze(-1)         # [B,K,P]
        recon_emb = self.emb_head(h)                        # [B,K,P,D]
        return mask_logits, recon_emb


In [5]:
class DiscreteTokenSlotSepModel(nn.Module):
    def __init__(self, s, window_size, vocab_size, num_slots, d_model, slot_iters, dec_hidden):
        super().__init__()
        self.s = s
        self.T = window_size
        self.N = s * s
        self.V = vocab_size
        self.K = num_slots
        self.D = d_model

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_time = nn.Parameter(torch.randn(1, window_size, 1, d_model) * 0.02)
        self.pos_space = nn.Parameter(torch.randn(1, 1, self.N, d_model) * 0.02)

        self.slot_attn = SlotAttention(num_slots=num_slots, dim=d_model, iters=slot_iters)
        self.decoder = SlotEmbeddingDecoder(num_slots, d_model, self.N, dec_hidden)

    def _emb_recon_loss(self, e_mix, e_gt, kind="cosine"):
        # e_mix, e_gt: [B,P,D]
        if kind == "mse":
            return F.mse_loss(e_mix, e_gt)
        elif kind == "cosine":
            # 1 - cosine
            e1 = F.normalize(e_mix, dim=-1)
            e2 = F.normalize(e_gt, dim=-1)
            return (1.0 - (e1 * e2).sum(dim=-1)).mean()
        else:
            raise ValueError("emb_loss must be 'mse' or 'cosine'")

    def _sampled_token_ce(self, e_mix, y, neg_samples=256, temperature=0.07):
        """
        e_mix: [B,P,D]  予測埋め込み（混合後）
        y:     [B,P]    正解トークンID
        負例をランダムサンプルして (1+M) クラス分類のCEを計算
        """
        B, P, D = e_mix.shape
        device = e_mix.device

        # [B*P]
        y_flat = y.reshape(-1)

        # negatives: [B*P, M] (uniform)
        neg = torch.randint(low=0, high=self.V, size=(y_flat.numel(), neg_samples), device=device)

        # candidates: [B*P, 1+M]
        cand = torch.cat([y_flat[:, None], neg], dim=1)

        # candidate embeddings: [B*P, 1+M, D]
        W = self.tok_emb.weight  # [V,D]
        cand_emb = W[cand]       # gather

        # logits: dot(e_mix, cand_emb)
        q = e_mix.reshape(-1, D)[:, None, :]               # [B*P,1,D]
        logits = (q * cand_emb).sum(dim=-1) / temperature  # [B*P, 1+M]

        # target index is always 0
        target = torch.zeros(y_flat.numel(), dtype=torch.long, device=device)
        return F.cross_entropy(logits, target)

    def forward(self, tokens, pos_sample=64, neg_samples=256, ce_weight=0.2,
                emb_loss="cosine", temperature=0.07, detach_slots=True, return_assign=False):
        """
        tokens: [B,T,N] long in [0..V-1]
        returns: loss, (optional hard_assign [B,T,P])
        """
        B, T, N = tokens.shape
        assert T == self.T and N == self.N

        # position sampling (shared across batch & time)
        if pos_sample is not None and pos_sample < N:
            pos_idx = torch.randperm(N, device=tokens.device)[:pos_sample]
            P = pos_sample
        else:
            pos_idx = None
            P = N

        # embeddings of tokens as "targets" in embedding space
        # e_gt_full: [B,T,N,D]
        e_gt_full = self.tok_emb(tokens)  # uses same embedding table (tying)
        x = e_gt_full + self.pos_time[:, :T] + self.pos_space  # [B,T,N,D]

        total_loss = 0.0
        prev_slots = None
        hard_list = [] if return_assign else None

        for t in range(T):
            # sample positions
            if pos_idx is not None:
                inp = x[:, t, pos_idx]            # [B,P,D]
                e_gt = e_gt_full[:, t, pos_idx]   # [B,P,D]
                y = tokens[:, t, pos_idx]         # [B,P]
            else:
                inp = x[:, t]                     # [B,N,D]
                e_gt = e_gt_full[:, t]            # [B,N,D]
                y = tokens[:, t]                  # [B,N]

            # slot inference
            slots = self.slot_attn(inp, prev_slots=prev_slots)  # [B,K,D]
            prev_slots = slots.detach() if detach_slots else slots

            # decode only sampled positions
            mask_logits, recon_emb_k = self.decoder(slots, pos_idx=pos_idx)  # [B,K,P,D], [B,K,P]
            masks = torch.softmax(mask_logits, dim=1)                         # [B,K,P]

            # mixture embedding: e_mix [B,P,D]
            e_mix = torch.sum(masks[:, :, :, None] * recon_emb_k, dim=1)

            # losses
            loss_emb = self._emb_recon_loss(e_mix, e_gt, kind=emb_loss)
            loss_ce  = self._sampled_token_ce(e_mix, y, neg_samples=neg_samples, temperature=temperature)

            total_loss = total_loss + (loss_emb + ce_weight * loss_ce)

            if return_assign:
                hard = torch.argmax(masks, dim=1)  # [B,P]
                hard_list.append(hard)

        loss = total_loss / T
        if return_assign:
            hard = torch.stack(hard_list, dim=1)  # [B,T,P]
            return loss, hard
        return loss


In [ ]:
# --- RawTokenDataset はあなたのものを使用 ---
dataset = RawTokenDataset(
    data_dir=cfg.data_dir,
    window_size=cfg.window_size,
    stride=cfg.stride,
    filter_interrupts=True,
    filter_overlaps=False,
)

s = dataset.metadata["s"]
vocab_size = int(dataset.metadata["vocab_size"])  # 262144
T = cfg.window_size
N = s * s

print("s, T, N, vocab_size =", s, T, N, vocab_size)

loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True, drop_last=True, num_workers=0)

model = DiscreteTokenSlotSepModel(
    s=s, window_size=T, vocab_size=vocab_size,
    num_slots=cfg.num_slots, d_model=cfg.d_model,
    slot_iters=cfg.slot_iters, dec_hidden=cfg.dec_hidden,
).to(cfg.device)

opt = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = GradScaler("cuda")

history = []
global_step = 0
opt.zero_grad(set_to_none=True)

for epoch in range(cfg.num_epochs):
    model.train()
    total = 0.0
    pbar = tqdm(loader, desc=f"epoch {epoch+1}/{cfg.num_epochs}")

    for batch in pbar:
        x = batch["input_ids"].to(cfg.device)  # [B, L]
        B = x.shape[0]
        x = x.view(B, T, N).long()

        with autocast("cuda"):
            loss = model(
                x,
                pos_sample=cfg.pos_sample,
                neg_samples=cfg.neg_samples,
                ce_weight=cfg.ce_weight,
                emb_loss=cfg.emb_loss,
                temperature=cfg.temperature,
                detach_slots=cfg.detach_slots,
                return_assign=False
            )
            loss = loss / cfg.accum_steps

        scaler.scale(loss).backward()
        global_step += 1

        if global_step % cfg.accum_steps == 0:
            scaler.unscale_(opt)
            if cfg.grad_clip and cfg.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)

        loss_val = float(loss.item() * cfg.accum_steps)
        total += loss_val
        pbar.set_postfix(loss=f"{loss_val:.4f}")

    avg = total / len(loader)
    history.append(avg)
    print(f"Epoch {epoch+1}: avg loss = {avg:.4f}")

plt.figure()
plt.plot(history, marker="o")
plt.grid(True)
plt.title("Train loss (emb recon + sampled CE)")
plt.show()


s, T, N, vocab_size = 16 6 256 262144


epoch 1/20:   3%|▎         | 34675/1343665 [34:45<18:36:49, 19.53it/s, loss=1.8762]

In [ ]:
@torch.no_grad()
def visualize_hard_assign_sampled(hard, s, pos_idx, Tshow=6):
    """
    hard: [T,P]  (sampled positions only)
    pos_idx: [P] indices in [0..N-1]
    """
    T, P = hard.shape
    Tshow = min(T, Tshow)
    plt.figure(figsize=(2*Tshow, 2))
    for t in range(Tshow):
        img = torch.full((s*s,), -1, dtype=torch.long)
        img[pos_idx.cpu()] = hard[t].cpu()
        img = img.view(s, s).numpy()
        plt.subplot(1, Tshow, t+1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"t={t}")
    plt.show()

model.eval()
batch = next(iter(loader))
x = batch["input_ids"].to(cfg.device)
B = x.shape[0]
x = x.view(B, T, N).long()

# 同じpos_idxを使うため、ここでpos_idxを固定生成
pos_idx = torch.randperm(N, device=cfg.device)[:cfg.pos_sample]

loss, hard = model(
    x,
    pos_sample=cfg.pos_sample,
    neg_samples=cfg.neg_samples,
    ce_weight=cfg.ce_weight,
    emb_loss=cfg.emb_loss,
    temperature=cfg.temperature,
    detach_slots=cfg.detach_slots,
    return_assign=True
)
print("loss:", float(loss.item()))
visualize_hard_assign_sampled(hard[0].cpu(), s=s, pos_idx=pos_idx.cpu(), Tshow=T)
